<a href="https://colab.research.google.com/github/kenedykimuhu-gif/dsc-course0-m8-lab/blob/main/Aviation_Accidents_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023.

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [69]:
!git clone https://github.com/kenedykimuhu-gif/dsc-course0-m8-lab.git
%cd dsc-course0-m8-lab

Cloning into 'dsc-course0-m8-lab'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 18 (delta 0), reused 0 (delta 0), pack-reused 15 (from 1)
Receiving objects: 100% (18/18), 4.91 MiB | 14.04 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/dsc-course0-m8-lab/dsc-course0-m8-lab/dsc-course0-m8-lab/dsc-course0-m8-lab/dsc-course0-m8-lab/dsc-course0-m8-lab/dsc-course0-m8-lab/dsc-course0-m8-lab


In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [71]:
df = pd.read_csv('data/AviationData.csv', encoding='latin-1')


/tmp/ipykernel_3564/1162149909.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/AviationData.csv', encoding='latin-1')


In [72]:
print("--- Datatypes and NaNs ---")
df.info()

--- Datatypes and NaNs ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make       

In [73]:
print("\n--- Summary Statistics ---")
display(df.describe())


--- Summary Statistics ---


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [74]:
print("--- Unique values in Amateur.Built ---")
print(df['Amateur.Built'].value_counts(dropna=False))

print("\n--- A look at Event.Date ---")
print(df['Event.Date'].head())

--- Unique values in Amateur.Built ---
Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64

--- A look at Event.Date ---
0    1948-10-24
1    1962-07-19
2    1974-08-30
3    1977-06-19
4    1979-08-02
Name: Event.Date, dtype: object


In [75]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df['Year'] = df['Event.Date'].dt.year

In [76]:
df_filtered = df[
    (df['Year'] >= 1983) &
    (df['Amateur.Built'].str.upper() == 'NO')
].copy()

print(f"Original rows: {len(df)}")
print(f"Filtered rows: {len(df_filtered)}")

Original rows: 88889
Filtered rows: 76960


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

### **CLEANING ASSUMPTIONS:**
1. 0 is used to fill missing injury values (NaNs), assuming there was no injuries reported.
2. To get the total passangers, we sum all injury categories.
3. To estimate the possibility of severe outcomes, we divide severe injuries by total passengers (`Fatal_Serious_Fraction`)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

**AIRCRAFT DAMAGE CLEANING:**
1. To ensure consistency, we strip whitecases and standardize text to uppercase.
2. For tracking total destruction rates, we created a derived binary column (`Is_Destroyed`).

In [77]:
print("--- Original Aircraft.damage values ---")
print(df_filtered['Aircraft.damage'].value_counts(dropna=False).head())

# Clean the text (uppercase and strip spaces)
df_filtered['Aircraft.damage'] = df_filtered['Aircraft.damage'].astype(str).str.upper().str.strip()

# The derived column
df_filtered['Is_Destroyed'] = np.where(df_filtered['Aircraft.damage'] == 'DESTROYED', 1, 0)

print("\n--- Is_Destroyed Counts (1 = Destroyed, 0 = Not Destroyed) ---")
print(df_filtered['Is_Destroyed'].value_counts())

--- Original Aircraft.damage values ---
Aircraft.damage
Substantial    55689
Destroyed      15469
NaN             3092
Minor           2594
Unknown          116
Name: count, dtype: int64

--- Is_Destroyed Counts (1 = Destroyed, 0 = Not Destroyed) ---
Is_Destroyed
0    61491
1    15469
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

## **MAKE CLEANING TASKS**
1. Standardize text by converting to uppercase and stripping whitespace.
2. For variations of same manufacturer to group together, remove corporate suffixes.
3. To avoid drawing conclusions from small sample sizes and to ensure robustness, filter the dataset and retain only makes with atleast 50 accidents.

In [78]:
# Inspect the raw Make column
print("--- Top 10 Raw Makes ---")
print(df_filtered['Make'].value_counts().head(10))

--- Top 10 Raw Makes ---
Make
Cessna    20818
Piper     11260
CESSNA     4922
Beech      4052
PIPER      2839
Bell       2006
Boeing     1498
BOEING     1151
BEECH      1042
Mooney     1029
Name: count, dtype: int64


In [79]:
import re

# Standardize text
df_filtered['Make'] = df_filtered['Make'].astype(str).str.upper().str.strip()

# Remove corporate suffixes using regex
df_filtered['Make'] = df_filtered['Make'].str.replace(r'\b(INC|CORP|CO|LTD|LLC|COMPANY|CORPORATION|AIRCRAFT|AIRPLANE)\b', '', regex=True)
df_filtered['Make'] = df_filtered['Make'].str.strip() # Strip again after removing suffixes

# Keep makes with at least 50 accidents
make_counts = df_filtered['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df_filtered = df_filtered[df_filtered['Make'].isin(valid_makes)].copy()

print("--- Cleaned Top 10 Makes ---")
print(df_filtered['Make'].value_counts().head(10))
print(f"\nFiltered shape: {df_filtered.shape}")

--- Cleaned Top 10 Makes ---
Make
CESSNA         25786
PIPER          14144
BEECH           5102
BOEING          2658
BELL            2594
MOONEY          1284
ROBINSON        1197
GRUMMAN         1065
BELLANCA         979
AIR TRACTOR      892
Name: count, dtype: int64

Filtered shape: (71035, 33)


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

## **Model Cleaning Tasks & Assumptions:**
1. Drop rows with NaNs.
2. Since model names are not globally unique, we create a derived column (`Make_Model`) by combining the make and model columns to serve an identifier for a specific type of plane.

In [80]:
# Inspecting the model column and NaNs
print(f"Number of NaNs in Model: {df_filtered['Model'].isna().sum()}")

# Looking at the most common models to check if they are unique
print("\n--- Top 10 Raw Models ---")
print(df_filtered['Model'].value_counts().head(10))

Number of NaNs in Model: 22

--- Top 10 Raw Models ---
Model
152          2230
172          1650
172N         1093
PA-28-140     863
172M          759
150           752
172P          665
182           617
180           596
150M          551
Name: count, dtype: int64


In [81]:
# Getting rid of NaNs
df_filtered = df_filtered.dropna(subset=['Model']).copy()

# Standardizing the text
df_filtered['Model'] = df_filtered['Model'].astype(str).str.upper().str.strip()

# Creating a unique identifier for a certain plane type
df_filtered['Make_Model'] = df_filtered['Make'] + " " + df_filtered['Model']

print("--- Top 10 Unique Make_Models ---")
print(df_filtered['Make_Model'].value_counts().head(10))
print(f"\nFiltered shape: {df_filtered.shape}")

--- Top 10 Unique Make_Models ---
Make_Model
CESSNA 152         2230
CESSNA 172         1650
CESSNA 172N        1093
PIPER PA-28-140     863
CESSNA 172M         759
CESSNA 150          752
CESSNA 172P         665
CESSNA 182          617
CESSNA 180          595
CESSNA 150M         551
Name: count, dtype: int64

Filtered shape: (71013, 34)


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks.

**Note**: You do not necessarily need to impute or drop NaNs here.

## **Cleaning tasks for other columns:**
1. Standardize the text for the columns.
2. For better visualizations, we group those with fewer than 50 occurences into an ('OTHER') category.
3. For ('Weather.Condition'), we only care about Visual (VMC) and Instrument (IMC) conditions; everything else is grouped as ('UNKNOWN').
4. Converted ('Number.of.Engines') to numeric and filled NaNs with the median.

In [82]:
cols_to_inspect = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']
for col in cols_to_inspect:
    print(f"--- {col} ---")
    print(df_filtered[col].value_counts(dropna=False).head(10))
    print("\n")

--- Engine.Type ---
Engine.Type
Reciprocating      55303
NaN                 5525
Turbo Shaft         3036
Turbo Prop          2869
Turbo Fan           2204
Unknown             1514
Turbo Jet            547
Geared Turbofan       12
LR                     1
UNK                    1
Name: count, dtype: int64


--- Weather.Condition ---
Weather.Condition
VMC    60977
IMC     5295
NaN     3802
UNK      704
Unk      235
Name: count, dtype: int64


--- Number.of.Engines ---
Number.of.Engines
1.0    54874
2.0     9680
NaN     4884
0.0      742
3.0      431
4.0      401
8.0        1
Name: count, dtype: int64


--- Purpose.of.flight ---
Purpose.of.flight
Personal              37129
Instructional          9448
Unknown                5518
NaN                    5435
Aerial Application     4202
Business               3451
Positioning            1460
Other Work Use         1069
Aerial Observation      716
Public Aircraft         651
Name: count, dtype: int64


--- Broad.phase.of.flight ---
Broad.ph

In [83]:
#  Engine Type
df_filtered['Engine.Type'] = df_filtered['Engine.Type'].astype(str).str.upper().str.strip()
engine_counts = df_filtered['Engine.Type'].value_counts()
df_filtered['Engine.Type'] = df_filtered['Engine.Type'].replace(engine_counts[engine_counts < 50].index, 'OTHER')

#  Weather Condition
df_filtered['Weather.Condition'] = df_filtered['Weather.Condition'].astype(str).str.upper().str.strip()
df_filtered['Weather.Condition'] = df_filtered['Weather.Condition'].apply(lambda x: x if x in ['VMC', 'IMC'] else 'UNKNOWN')

#  Number of Engines
df_filtered['Number.of.Engines'] = pd.to_numeric(df_filtered['Number.of.Engines'], errors='coerce')
df_filtered['Number.of.Engines'] = df_filtered['Number.of.Engines'].fillna(df_filtered['Number.of.Engines'].median())

#  Purpose of flight
df_filtered['Purpose.of.flight'] = df_filtered['Purpose.of.flight'].astype(str).str.upper().str.strip()
purpose_counts = df_filtered['Purpose.of.flight'].value_counts()
df_filtered['Purpose.of.flight'] = df_filtered['Purpose.of.flight'].replace(purpose_counts[purpose_counts < 50].index, 'OTHER')

#  Broad phase of flight
df_filtered['Broad.phase.of.flight'] = df_filtered['Broad.phase.of.flight'].astype(str).str.upper().str.strip()
phase_counts = df_filtered['Broad.phase.of.flight'].value_counts()
df_filtered['Broad.phase.of.flight'] = df_filtered['Broad.phase.of.flight'].replace(phase_counts[phase_counts < 50].index, 'OTHER')

print("Other columns cleaned successfully.")

Other columns cleaned successfully.


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [84]:
#  Inspecting percentage of missing values in each column
missing_pct = df_filtered.isnull().sum() / len(df_filtered)

# Identifying columns with more than 50% missing data
cols_to_drop = missing_pct[missing_pct > 0.5].index.tolist()

print(f"Dropping {len(cols_to_drop)} columns with >50% missing values:")
print(cols_to_drop)

# Dropping those columns
df_filtered = df_filtered.drop(columns=cols_to_drop)

print(f"\nShape after dropping columns: {df_filtered.shape}")

Dropping 6 columns with >50% missing values:
['Latitude', 'Longitude', 'Aircraft.Category', 'FAR.Description', 'Schedule', 'Air.carrier']

Shape after dropping columns: (71013, 28)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [85]:
df_filtered.to_csv('data/cleaned_aviation_data.csv', index=False)

In [86]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [87]:
df_filtered.to_csv('/content/drive/MyDrive/cleaned_aviation_data.csv', index=False)
print("Saved to Google Drive!")

Saved to Google Drive!
